In [132]:
from sklearn.compose import make_column_transformer

In [133]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import SGDClassifier

In [134]:
dataset = sns.load_dataset("titanic")

dataset.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [135]:
x = dataset.drop(columns=['survived', 'class', 'embark_town', 'alive', 'deck', 'who', 'sibsp', 'adult_male'], axis=1)
y = dataset['survived']
x

,pclass,sex,age,parch,fare,embarked,alone
0,3,male,22.0,0,7.2500,S,False
1,1,female,38.0,0,71.2833,C,False
2,3,female,26.0,0,7.9250,S,True
3,1,female,35.0,0,53.1000,S,False
4,3,male,35.0,0,8.0500,S,True
...,...,...,...,...,...,...,...
886,2,male,27.0,0,13.0000,S,True
887,1,female,19.0,0,30.0000,S,True
888,3,female,NaN,2,23.4500,S,False
889,1,male,26.0,0,30.0000,C,True


In [136]:
X_train, X_test, Y_train, Y_test = train_test_split(x, y, train_size=0.2)

In [137]:
onehot = ['sex', 'embarked', 'alone']
robustscal = ['age', 'fare', 'parch', 'pclass']

In [138]:
pipeline_robstscaler = make_pipeline(SimpleImputer(), RobustScaler())
pipeline_onehot = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder())

preprocessor = make_column_transformer(
                            (pipeline_onehot, onehot),
                            (pipeline_robstscaler, robustscal)
                        )
model = make_pipeline(preprocessor, SGDClassifier())
model.fit(X_train, Y_train)
param_grid = {
    'sgdclassifier__loss': ['hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'],
    'sgdclassifier__penalty': ['l2', 'l1', 'elasticnet', None],
    'sgdclassifier__alpha': np.arange(0.001, 0.99),
    'sgdclassifier__learning_rate': ['constant', 'optimal', 'adaptive']
    }
grid = GridSearchCV(model, param_grid, cv=5)

grid.fit(X_train, Y_train)

grid.best_params_

/Users/fabiochaput/Documents/VS/portfolio/CardFraud/venv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
360 fits failed out of a total of 540.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/fabiochaput/Documents/VS/portfolio/CardFraud/venv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/fabiochaput/Documents/VS/portfolio/CardFraud/venv/lib/python3.11/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^

{'sgdclassifier__alpha': 0.001,
 'sgdclassifier__learning_rate': 'optimal',
 'sgdclassifier__loss': 'huber',
 'sgdclassifier__penalty': 'elasticnet'}